# 🥛 Dairy Sales Forecasting — Model Training
### Phase 1: Prophet + SARIMAX | Festival-Aware | Multi-Product

**Flow:**
1. Install & Import
2. Load & Validate Data
3. Build Festival Calendar
4. Train Prophet (per product)
5. Train SARIMAX (per product)
6. Evaluate & Compare
7. Query: Any Festival → Product Spike Prediction

> Upload `final_sales_with_festivals.csv` when prompted in Cell 3

## 📦 Cell 1 — Install Dependencies

In [1]:
%%capture
!pip install prophet statsmodels scikit-learn pandas numpy openpyxl
print('✅ All packages installed')

## 📥 Cell 2 — Imports

In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import pickle, os, json
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from google.colab import files

os.makedirs('saved_models/prophet',  exist_ok=True)
os.makedirs('saved_models/sarimax',  exist_ok=True)
os.makedirs('outputs',               exist_ok=True)

print('✅ Imports done')

ModuleNotFoundError: No module named 'google.colab'

## 📂 Cell 3 — Upload & Load Data

In [ ]:
# ── Upload CSV ──
uploaded = files.upload()   # select final_sales_with_festivals.csv
CSV_PATH = list(uploaded.keys())[0]
print(f'Uploaded: {CSV_PATH}')

# ── CONFIG ──
TEST_DAYS = 90    # last N days held out for evaluation
MIN_ROWS  = 60    # skip products with fewer rows

# ── Load ──
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
print('Columns found:', df.columns.tolist())
df.head(3)

## 🧹 Cell 4 — Clean & Validate

In [ ]:
# ── Rename to standard names ──
df = df.rename(columns={
    'Date':     'ds',
    'Quantity': 'y',
    'Product':  'product',
    'Festival': 'festival'
})

# ── FIX 1: Ensure ds is datetime (fixes 'str has no attribute year') ──
df['ds'] = pd.to_datetime(df['ds'], errors='coerce', dayfirst=True)
bad_dates = df['ds'].isna().sum()
if bad_dates > 0:
    print(f'  ⚠ Dropped {bad_dates} rows with unparseable dates')

df['festival'] = df['festival'].fillna('').astype(str).str.strip()
df['y']        = pd.to_numeric(df['y'], errors='coerce')
df = df.dropna(subset=['y', 'ds']).sort_values('ds').reset_index(drop=True)

# ── Product filter ──
prod_counts = df['product'].value_counts()
products    = prod_counts[prod_counts >= MIN_ROWS].index.tolist()

print(f'Total rows       : {len(df):,}')
print(f'Total products   : {df["product"].nunique()}')
print(f'Trainable (≥{MIN_ROWS}r): {len(products)}')
print(f'Date range       : {df["ds"].min().date()} → {df["ds"].max().date()}')
print(f'Unique festivals : {df[df["festival"] != ""]["festival"].nunique()}')

# ── Baseline: normal days (no festival) ──
non_fest  = df[df['festival'] == '']
baselines = (
    non_fest.groupby('product')['y']
    .agg(baseline_mean='mean', baseline_std='std')
    .round(2)
)
baselines.to_csv('outputs/product_baselines.csv')
print('\n✅ Baselines saved')
baselines.head()

## 📅 Cell 5 — Build Festival Calendar

In [ ]:
# ── Prophet holiday dataframe ──
# FIX: ds must already be datetime (done in Cell 4)
fest_raw = (
    df[df['festival'] != ''][['ds', 'festival']]
    .drop_duplicates()
    .copy()
)

# One row per (date, festival) — Prophet format
fest_df = fest_raw.rename(columns={'festival': 'holiday'})
fest_df['lower_window'] = -2   # effect starts 2 days before
fest_df['upper_window'] =  2   # effect ends 2 days after
fest_df = fest_df.reset_index(drop=True)

known_festivals = sorted(fest_df['holiday'].unique().tolist())
with open('outputs/known_festivals.json', 'w') as f:
    json.dump(known_festivals, f, indent=2)

# ── SARIMAX exogenous dummies ──
# Binary matrix: rows=dates, cols=festivals
all_dates  = pd.date_range(df['ds'].min(), df['ds'].max(), freq='D')
exog_all   = pd.DataFrame(0, index=all_dates, columns=known_festivals)
exog_all.index.name = 'ds'

for fest in known_festivals:
    fest_dates = fest_raw[fest_raw['festival'] == fest]['ds'].values
    for fd in fest_dates:
        mask = (all_dates >= fd - pd.Timedelta(days=2)) & \
               (all_dates <= fd + pd.Timedelta(days=2))
        exog_all.loc[mask, fest] = 1

exog_all.to_csv('outputs/festival_exog_matrix.csv')

print(f'Known festivals   : {len(known_festivals)}')
print(f'Prophet holiday df: {len(fest_df)} rows')
print(f'SARIMAX exog shape: {exog_all.shape}')
print('\nFestivals:', known_festivals)
print('\n✅ Festival calendar ready')

---
## 🔮 Cell 6 — Train Prophet (All Products)
> **What Prophet learns:** yearly + weekly seasonality + holiday effects.  
> **Spike extraction:** each holiday gets a learned % effect (multiplicative mode).

In [ ]:
import time

prophet_metrics   = []
prophet_spikes    = {}   # product → {festival → spike_%}
prophet_forecasts = {}   # product → forecast df

print(f'Training {len(products)} Prophet models...\n')
print(f'{"Product":<40} {"MAE":>8} {"MAPE%":>8} {"RMSE":>8} {"Grade":>6}')
print('-' * 72)

for product in products:
    df_p = df[df['product'] == product][['ds', 'y']].dropna().copy()

    # ── FIX 2: Guard against empty test set ──
    data_end = df_p['ds'].max()
    cutoff   = data_end - pd.Timedelta(days=TEST_DAYS)
    train_df = df_p[df_p['ds'] <= cutoff]
    test_df  = df_p[df_p['ds']  > cutoff]

    if len(train_df) < 30 or len(test_df) == 0:
        print(f'  {product[:40]:<40} SKIPPED (train={len(train_df)}, test={len(test_df)})')
        continue

    try:
        m = Prophet(
            holidays           = fest_df,
            yearly_seasonality = True,
            weekly_seasonality = True,
            daily_seasonality  = False,
            seasonality_mode   = 'multiplicative',
            interval_width     = 0.90,
        )
        m.add_country_holidays(country_name='IN')
        m.fit(train_df)

        future   = m.make_future_dataframe(periods=TEST_DAYS)
        forecast = m.predict(future)
        prophet_forecasts[product] = forecast

        # ── Metrics on test set ──
        test_fc = forecast[forecast['ds'].isin(test_df['ds'])][['ds','yhat']]
        merged  = test_df.merge(test_fc, on='ds')

        if len(merged) == 0:   # dates don't align (daily gaps in data)
            merged = test_df.copy()
            merged['yhat'] = forecast.iloc[-len(test_df):]['yhat'].values

        mae  = mean_absolute_error(merged['y'], merged['yhat'])
        mape = mean_absolute_percentage_error(merged['y'], merged['yhat']) * 100
        rmse = float(np.sqrt(((merged['y'] - merged['yhat']) ** 2).mean()))
        grade = 'A' if mape < 15 else ('B' if mape < 25 else 'C')

        # ── Extract holiday spike effect ──
        # In multiplicative mode, holidays column = fractional lift
        # e.g. 0.25 → 25% above baseline
        spike_map = {}
        if 'holidays' in forecast.columns:
            holiday_fc = forecast.merge(
                fest_df[['ds','holiday']].drop_duplicates(),
                on='ds', how='left'
            )
            for fest_name, grp in holiday_fc.dropna(subset=['holiday']).groupby('holiday'):
                spike_map[fest_name] = round(float(grp['holidays'].mean()) * 100, 2)
        prophet_spikes[product] = spike_map

        # ── Save model ──
        safe = product.replace('/', '_').replace(' ', '_')[:40]
        with open(f'saved_models/prophet/{safe}.pkl', 'wb') as f:
            pickle.dump({'model': m, 'train': train_df, 'test': test_df,
                         'forecast': forecast}, f)

        prophet_metrics.append({
            'Product':    product,
            'Train_Rows': len(train_df),
            'Test_Rows':  len(test_df),
            'MAE':        round(mae, 2),
            'RMSE':       round(rmse, 2),
            'MAPE_%':     round(mape, 2),
            'Grade':      grade
        })
        print(f'  {product[:40]:<40} {mae:>8.1f} {mape:>7.1f}% {rmse:>8.1f} {grade:>6}')

    except Exception as e:
        print(f'  {product[:40]:<40} FAILED: {e}')

# ── Save Prophet outputs ──
prophet_metrics_df = pd.DataFrame(prophet_metrics).sort_values('MAPE_%')
prophet_metrics_df.to_csv('outputs/prophet_metrics.csv', index=False)

prophet_spike_df = pd.DataFrame(prophet_spikes).T
prophet_spike_df.index.name = 'Product'
prophet_spike_df.to_csv('outputs/prophet_festival_spikes.csv')

print(f'\n✅ Prophet done: {len(prophet_metrics)} models trained')
print(f'   Avg MAPE: {prophet_metrics_df["MAPE_%"].mean():.1f}%')
prophet_metrics_df

---
## 📈 Cell 7 — Train SARIMAX (All Products)
> **What SARIMAX learns:** AR/MA patterns + seasonal period + festival coefficients.  
> **Spike extraction:** each festival dummy's coefficient = extra units sold per day.

In [ ]:
# ── SARIMAX config ──
# (p,d,q) non-seasonal | (P,D,Q,s) seasonal with s=7 (weekly pattern)
SARIMAX_ORDER    = (1, 1, 1)
SARIMAX_SEASONAL = (1, 1, 1, 7)

def get_d(series):
    """ADF test → d=0 if stationary else d=1."""
    try:
        return 0 if adfuller(series.dropna())[1] < 0.05 else 1
    except:
        return 1

sarimax_metrics  = []
sarimax_coeffs   = {}   # product → {festival → coefficient}

print(f'Training {len(products)} SARIMAX models...\n')
print(f'{"Product":<40} {"MAE":>8} {"MAPE%":>8} {"RMSE":>8} {"d":>4} {"Grade":>6}')
print('-' * 76)

for product in products:
    df_p = (
        df[df['product'] == product][['ds', 'y']]
        .dropna()
        .set_index('ds')
        .asfreq('D')           # fill daily freq
        .fillna(method='ffill')# forward-fill gaps
    )

    cutoff    = df_p.index.max() - pd.Timedelta(days=TEST_DAYS)
    train_ser = df_p[df_p.index <= cutoff]['y']
    test_ser  = df_p[df_p.index >  cutoff]['y']

    # ── FIX 2: Guard empty test ──
    if len(train_ser) < 30 or len(test_ser) == 0:
        print(f'  {product[:40]:<40} SKIPPED (train={len(train_ser)}, test={len(test_ser)})')
        continue

    # ── Align exog to this product's dates ──
    train_exog = exog_all.reindex(train_ser.index).fillna(0)
    test_exog  = exog_all.reindex(test_ser.index).fillna(0)

    # Only keep festival columns active in training period
    active_cols = train_exog.columns[train_exog.sum() > 0].tolist()
    train_exog  = train_exog[active_cols] if active_cols else None
    test_exog_a = test_exog[active_cols]  if active_cols else None

    d_auto = get_d(train_ser)
    order  = (SARIMAX_ORDER[0], d_auto, SARIMAX_ORDER[2])

    try:
        model = SARIMAX(
            train_ser,
            exog                 = train_exog,
            order                = order,
            seasonal_order       = SARIMAX_SEASONAL,
            enforce_stationarity = False,
            enforce_invertibility= False,
        )
        result = model.fit(disp=False, maxiter=100)

        preds = result.forecast(steps=len(test_ser), exog=test_exog_a)
        preds = preds.clip(lower=0)

        mae  = mean_absolute_error(test_ser, preds)
        mape = mean_absolute_percentage_error(test_ser, preds) * 100
        rmse = float(np.sqrt(((test_ser.values - preds.values) ** 2).mean()))
        grade = 'A' if mape < 15 else ('B' if mape < 25 else 'C')

        # ── Extract festival coefficients ──
        # Coefficient = extra units sold per day when that festival dummy = 1
        coeffs = {}
        if active_cols:
            for col in active_cols:
                if col in result.params.index:
                    coeffs[col] = round(float(result.params[col]), 4)
        sarimax_coeffs[product] = coeffs

        # ── Save model ──
        safe = product.replace('/', '_').replace(' ', '_')[:40]
        with open(f'saved_models/sarimax/{safe}.pkl', 'wb') as f:
            pickle.dump({
                'result':         result,
                'active_cols':    active_cols,
                'order':          order,
                'seasonal_order': SARIMAX_SEASONAL,
                'train_end':      train_ser.index.max()
            }, f)

        sarimax_metrics.append({
            'Product':    product,
            'Train_Rows': len(train_ser),
            'Test_Rows':  len(test_ser),
            'd_used':     d_auto,
            'AIC':        round(result.aic, 2),
            'MAE':        round(mae, 2),
            'RMSE':       round(rmse, 2),
            'MAPE_%':     round(mape, 2),
            'Grade':      grade
        })
        print(f'  {product[:40]:<40} {mae:>8.1f} {mape:>7.1f}% {rmse:>8.1f} {d_auto:>4}  {grade:>6}')

    except Exception as e:
        print(f'  {product[:40]:<40} FAILED: {e}')

# ── Save SARIMAX outputs ──
sarimax_metrics_df = pd.DataFrame(sarimax_metrics).sort_values('MAPE_%')
sarimax_metrics_df.to_csv('outputs/sarimax_metrics.csv', index=False)

sarimax_coeff_df = pd.DataFrame(sarimax_coeffs).T
sarimax_coeff_df.index.name = 'Product'
sarimax_coeff_df.to_csv('outputs/sarimax_festival_coefficients.csv')

print(f'\n✅ SARIMAX done: {len(sarimax_metrics)} models trained')
print(f'   Avg MAPE: {sarimax_metrics_df["MAPE_%"].mean():.1f}%')
sarimax_metrics_df

---
## 📊 Cell 8 — Compare Prophet vs SARIMAX

In [ ]:
p = prophet_metrics_df.rename(columns={'MAPE_%':'P_MAPE','MAE':'P_MAE','RMSE':'P_RMSE','Grade':'P_Grade'})
s = sarimax_metrics_df.rename(columns={'MAPE_%':'S_MAPE','MAE':'S_MAE','RMSE':'S_RMSE','Grade':'S_Grade'})

comp = p[['Product','P_MAE','P_RMSE','P_MAPE','P_Grade']].merge(
       s[['Product','S_MAE','S_RMSE','S_MAPE','S_Grade']], on='Product', how='inner')

comp['Winner']    = comp.apply(lambda r: 'Prophet' if r['P_MAPE'] < r['S_MAPE'] else 'SARIMAX', axis=1)
comp['MAPE_diff'] = (comp['P_MAPE'] - comp['S_MAPE']).round(2)

# ── Ensemble weights (inverse-MAPE) ──
def inv_mape_weights(p_mape, s_mape):
    p_mape = max(p_mape, 0.01)
    s_mape = max(s_mape, 0.01)
    pw = (1/p_mape) / (1/p_mape + 1/s_mape)
    return round(pw, 3), round(1-pw, 3)

comp[['Prophet_W','SARIMAX_W']] = comp.apply(
    lambda r: pd.Series(inv_mape_weights(r['P_MAPE'], r['S_MAPE'])), axis=1)

comp.to_csv('outputs/model_comparison.csv', index=False)

# ── Print summary ──
p_wins = (comp['Winner'] == 'Prophet').sum()
s_wins = (comp['Winner'] == 'SARIMAX').sum()
print('=' * 60)
print(f'  Products compared   : {len(comp)}')
print(f'  Prophet wins (MAPE) : {p_wins} ({p_wins/len(comp)*100:.0f}%)')
print(f'  SARIMAX wins (MAPE) : {s_wins} ({s_wins/len(comp)*100:.0f}%)')
print(f'  Avg MAPE — Prophet  : {comp["P_MAPE"].mean():.2f}%')
print(f'  Avg MAPE — SARIMAX  : {comp["S_MAPE"].mean():.2f}%')
print(f'  Overall winner      : {"Prophet" if p_wins > s_wins else "SARIMAX"}')
print('=' * 60)
print('\nPer-product result:')
comp[['Product','P_MAPE','S_MAPE','Winner','Prophet_W','SARIMAX_W']]

---
## 🎉 Cell 9 — Festival Spike Query
### Manager picks ANY festival → which products spike & by how much

In [ ]:
def predict_festival_spike(festival_name: str, top_n: int = 10, model: str = 'both'):
    """
    festival_name : any string — seen or unseen
    top_n         : how many top products to show
    model         : 'prophet', 'sarimax', or 'both' (ensemble)

    SEEN festivals   → exact learned effect
    UNSEEN festivals → average of all known festival effects (conservative prior)
    """
    baselines_local = pd.read_csv('outputs/product_baselines.csv', index_col=0)
    known           = json.load(open('outputs/known_festivals.json'))
    is_seen         = festival_name in known
    source_tag      = 'learned' if is_seen else f'estimated (avg of {len(known)} known festivals)'

    rows = []

    # ── Prophet spikes ──
    p_spikes = pd.read_csv('outputs/prophet_festival_spikes.csv', index_col=0)
    for product in p_spikes.index:
        bm = baselines_local.loc[product, 'baseline_mean'] if product in baselines_local.index else None
        if is_seen and festival_name in p_spikes.columns:
            p_pct = float(p_spikes.loc[product, festival_name])
        else:
            p_pct = float(p_spikes.loc[product].dropna().mean() or 0)
        rows.append({'Product': product, 'Prophet_Spike_%': round(p_pct, 2),
                     'Baseline': bm})

    df_rows = pd.DataFrame(rows).set_index('Product')

    # ── SARIMAX spikes ──
    s_coeffs = pd.read_csv('outputs/sarimax_festival_coefficients.csv', index_col=0)
    s_pcts   = {}
    for product in s_coeffs.index:
        bm = baselines_local.loc[product, 'baseline_mean'] if product in baselines_local.index else 1
        if is_seen and festival_name in s_coeffs.columns:
            coeff = float(s_coeffs.loc[product, festival_name])
        else:
            coeff = float(s_coeffs.loc[product].dropna().mean() or 0)
        s_pcts[product] = round((coeff / bm) * 100, 2) if bm else 0
    df_rows['SARIMAX_Spike_%'] = pd.Series(s_pcts)

    # ── Ensemble (weighted avg using model comparison weights) ──
    try:
        weights = pd.read_csv('outputs/model_comparison.csv').set_index('Product')
        df_rows['Ensemble_Spike_%'] = df_rows.apply(
            lambda r: round(
                r['Prophet_Spike_%'] * weights.loc[r.name, 'Prophet_W'] +
                r['SARIMAX_Spike_%'] * weights.loc[r.name, 'SARIMAX_W']
            if r.name in weights.index else (r['Prophet_Spike_%'] + r['SARIMAX_Spike_%']) / 2, 2),
            axis=1
        )
    except:
        df_rows['Ensemble_Spike_%'] = ((df_rows['Prophet_Spike_%'] + df_rows['SARIMAX_Spike_%']) / 2).round(2)

    df_rows['Extra_Units_Est'] = (df_rows['Baseline'] * df_rows['Ensemble_Spike_%'] / 100).round(1)
    df_rows['Source']          = source_tag
    result = df_rows.sort_values('Ensemble_Spike_%', ascending=False).head(top_n)

    print(f"\n{'='*70}")
    print(f"  Festival : '{festival_name}'  |  Status: {'✅ Seen (trained)' if is_seen else '⚡ New/Unseen (estimated)'}")
    print(f"  Source   : {source_tag}")
    print(f"{'='*70}")
    print(f"  {'Product':<38} {'P_Spike%':>9} {'S_Spike%':>9} {'Ensemble%':>10} {'Extra Units':>12}")
    print('  ' + '-'*80)
    for prod, row in result.iterrows():
        print(f"  {prod[:38]:<38} {row['Prophet_Spike_%']:>8.1f}% "
              f"{row['SARIMAX_Spike_%']:>8.1f}% {row['Ensemble_Spike_%']:>9.1f}% "
              f"{str(row['Extra_Units_Est']):>12}")
    return result.reset_index()

print('✅ predict_festival_spike() ready')
print('Usage examples in next cell ↓')

## 🔍 Cell 10 — Run Queries

In [ ]:
# ── Example 1: Seen festival ──
# result = predict_festival_spike('Diwali', top_n=10)

# ── Example 2: New festival (manager picks from dropdown/UI later) ──
# result = predict_festival_spike('New Year', top_n=10)

# ── Example 3: Any custom name ──
# result = predict_festival_spike('Pongal', top_n=5)

# ── Run all known festivals at once → summary table ──
known_fests = json.load(open('outputs/known_festivals.json'))
print(f'Known festivals in your data ({len(known_fests)}):')
for i, f in enumerate(known_fests, 1):
    print(f'  {i:2}. {f}')

print('\n→ Uncomment one of the examples above and run')

## 💾 Cell 11 — Download All Outputs

In [ ]:
import shutil

# Zip everything and download
shutil.make_archive('model_training_outputs', 'zip', 'outputs')
files.download('model_training_outputs.zip')

print('\nOutputs in the zip:')
print('  prophet_metrics.csv              ← MAE/MAPE/RMSE/Grade per product')
print('  sarimax_metrics.csv              ← MAE/MAPE/RMSE/AIC/Grade per product')
print('  prophet_festival_spikes.csv      ← % spike per festival per product (Prophet)')
print('  sarimax_festival_coefficients.csv← learned coeff per festival (SARIMAX)')
print('  model_comparison.csv             ← side-by-side + ensemble weights')
print('  product_baselines.csv            ← normal day mean/std per product')
print('  known_festivals.json             ← festival list')
print('  festival_exog_matrix.csv         ← 0/1 dummy matrix for SARIMAX')
print('\nNote: .pkl model files are in saved_models/ — download separately if needed')